# Rule: **build_industrial_production_per_country_tomorrow**


**Description**

The future industrial production per country is built using today's industrial production and applies scenario-specific scaling factors only for the steel, alumina and HVC sectors, with parameters from the configuration file according to the horizon year. However, the total production for these sectors remains intact. The remaining industrial sectors maintain their present production levels.

The configuration parameters that determine the future industrial production are defined under the **industry** section of the config file:  
- industry.St_primary_fraction
- industry.DRI_fraction
- industry.Al_primary_fraction
- industry.HVC_primary_fraction
- industry.HVC_mechanical_recycling_fraction
- industry.HVC_chemical_recycling_fraction

**Inputs**

- resources/{prefix}/{name}/`industrial_production_per_country.csv`

**Outputs**

- resources/{prefix}/{name}/`industrial_production_per_country_tomorrow_{horizon}.csv`

In [ ]:
######################################## Parameters

### Run
prefix = ''
name = ''

### Network
clusters = '' # number of clusters or 'adm'
opts = ''
sector_opts = ''
horizon = ''

In [ ]:
##### Imports
import pandas as pd
import os 
import sys
import matplotlib.pyplot as plt
import numpy as np
import plotly.graph_objects as go

##### Import local functions
sys.path.append(os.path.abspath(os.path.join('..')))
import functions as xp

##### Read params.yaml
params = xp.read_params('../params.yaml')

##### Ignore warnings
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

##### Set options
pd.set_option("display.max_columns", None)

## `industrial_production_per_country_tomorrow_{horizon}.csv`  
Load the file and preview its content.

In [ ]:
file = f"industrial_production_per_country_tomorrow_{horizon}.csv"

ind_prod_tomorrow = xp.load_file_csv(
    params,
    file,
    prefix=prefix,
    name=name,
    location="resources",
)

ind_prod_tomorrow.head()

See the future industrial production for a specific country

In [ ]:
# Select the country
country_code = "ES"

# Filter by country
ind_prod_tomorrow_country = ind_prod_tomorrow[ind_prod_tomorrow.iloc[:, 0] == country_code].iloc[0]
ind_prod_tomorrow_country

What is the difference between current production and the expected production for the horizon year in the specified country?

In [ ]:
# Load present industrial production for comparison
file = f"industrial_production_per_country.csv"

ind_prod_today = xp.load_file_csv(
    params,
    file,
    prefix=prefix,
    name=name,
    location="resources",
)

ind_prod_today_country = ind_prod_today[ind_prod_today.iloc[:, 0] == country_code].iloc[0]

# Build comparison DataFrame
df = pd.DataFrame({
    "today": pd.to_numeric(ind_prod_today_country, errors="coerce"),
    horizon: pd.to_numeric(ind_prod_tomorrow_country, errors="coerce"),
})

# Remove posibles NaN
df = df.dropna()

###### Plot
fig = go.Figure()

##### Select vertical or horizontal bars

fig.add_trace(go.Bar(
    ################# Vertical bars
    x=df.index,
    y=df["today"],
    orientation="v",
    ################# Horizontal bars
    # y=df.index,
    # x=df["today"],
    # orientation="h",
    name="Today",
))

fig.add_trace(go.Bar(
    ################# Vertical bars
    x=df.index,
    y=df[horizon],
    orientation="v",
    ################# Horizontal bars
    # y=df.index,
    # x=df[horizon],
    # orientation="h",
    name=str(horizon),
))

# Layout settings
fig.update_layout(
    title=f"Industrial production by sector: today vs {horizon}",
    ################# Vertical bars
    yaxis_title="Industrial production [kton/a]",
    xaxis_title="Sector",
    ################# Horizontal bars
    # xaxis_title="Industrial production [kton/a]",
    # yaxis_title="Sector",
    barmode="group",
    font=dict(size=16),
    height=600,
    width=1200,
    plot_bgcolor="white",
    paper_bgcolor="white",
)

fig.show()